# 1. 简介
本文是作者在学习完朴素贝叶斯算法后进行的第一个机器学习项目，旨在完整地探索一遍机器学习的基本步骤，包括数据的预处理，特征工程，模型训练以及模型评估。

朴素贝叶斯本身的理论并不复杂，本人特地写了一篇文章：[用一个例子带你理解朴素贝叶斯分类算法](https://zhuanlan.zhihu.com/p/2023761737086682477)

在正式开始使用该算法前，我更希望读者先理清楚朴素贝叶斯算法的基本思想以及其局限性。这为我们后续每一步的操作提供了实际的指导。

# 2. 数据预处理

## 2.1 样本不平衡问题

In [7]:
import pandas as pd


train = pd.read_csv(r"./data/train.csv")
test = pd.read_csv(r"./data/train.csv")

In [8]:
train.groupby("Label").agg(数量=("ID", "count"))

,数量
Label,
0,621
1,3873
2,570
3,6478
4,6344
5,1641
6,2881
7,1332
8,992


可以看到，不同标签之间的类别极端不平衡。在这种情况下训练模型，会有什么影响呢？我们看朴素贝叶斯中后验概率的计算方法：
$$
\begin{aligned}
&P(Y = c_k | X^{(1)} = k_1, X^{(2)} = k_2,..., X^{(n)} = k_n) \\
&= \frac{P(X^{(1)} = k_1, X^{(2)} = k_2,..., X^{(n)} = k_n | Y = c_k) \cdot P(Y = c_k)}{P(X^{(1)} = k_1, X^{(2)} = k_2,..., X^{(n)} = k_n)} \\
&= \frac{[P(X^{(1)} = k_1 | Y = c_k) \cdot P(X^{(2)} = k_2 | Y = c_k) \cdot ... \cdot P(X^{(n)} = k_n | Y = c_k)] \cdot P(Y = c_k)}{P(X^{(1)} = k_1, X^{(2)} = k_2,..., X^{(n)} = k_n)}
\end{aligned}
$$

不同标签的样本量实际上有两层影响：
1. 影响了先验概率$P(Y = c_k)$，这会使得模型天然偏向多数类；
2. 影响少数类中，$P(X^{(m)} = k_m | Y = c_k)$的估计稳定性；

根据我个人的理解：
1. 对于第一个影响，如果样本收集是合理的（即训练集与真实总体分布一致），那么先验偏差不是错误，而是反映了真实情况。此时模型偏向多数类是合理的；
2. 对于第二个影响，如果各个类别的样本量足够大，那么也不构成问题；

为了方便，我们假设这种先验分布是大体合理的，因此类别不均衡不是我们的主要矛盾；而我们少数类的样本量非常少，如10和12，仅有100个左右的样本，估计稳定性很差，这才是我们的主要矛盾！

一种典型的不可靠做法是对少数类进行上采样。然后，上采样本质这是复制自身的信息，并不会增加真正的统计显著性；另外一种方法是合并少数类，或将少数类并到语义上相近的类别。不过鉴于我们这是一个比赛型项目，我们假设不能增删原始类别，因此这一方案也被否决。

我认为最可靠的方法是“数据增强”，因为我们的数据是文本，我们可以使用自然语言增强技术为极少数类生成新样本：
- 回译：将少数类样本翻译成另一种语言（如中文→英文→中文），生成语义相似的变体。
- 同义词替换：随机替换句子中的某些词为同义词（使用WordNet或词向量）。
- 随机插入/删除：非核心词的随机增删。

In [5]:
train

,ID,content,Label
0,tweet_0,@tiffanylue i know i was listenin to bad habi...,0
1,tweet_1,Layin n bed with a headache ughhhh...waitin o...,1
2,tweet_2,Funeral ceremony...gloomy friday...,1
3,tweet_3,wants to hang out with friends SOON!,2
4,tweet_4,@dannycastillo We want to trade with someone w...,3
...,...,...,...
29995,tweet_29995,@shonali I think the lesson of the day is not ...,3
29996,tweet_29996,@lovelylisaj can you give me the link for the ...,3
29997,tweet_29997,"@sendsome2me haha, yeah. Twitter has many uses...",3
29998,tweet_29998,Happy Mother's Day to all the mommies out ther...,6
